# Power Demand Forecasting

## Project Overview

Forecasts **daily power demand** (MW) over a 14-day horizon. Synthetic data spans 2 years with weekly patterns, summer/winter peaks, and holiday dips.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily power demand, predict the next 14 days. Power grid operators need demand forecasts for generation scheduling, fuel procurement, and grid stability.

## Why This Project Matters

Power grids must balance supply and demand in real-time. Under-forecasting leads to blackouts; over-forecasting wastes fuel and increases costs. Accurate day-ahead forecasting is the backbone of grid operations.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily power demand
- Weekly pattern (higher weekdays, lower weekends)
- Summer peak (air conditioning) and winter peak (heating)
- Holiday dips
- Slight upward trend

| Column | Description |
|--------|-------------|
| `date` | Date |
| `power_mw` | Daily average power demand (MW) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'power_mw'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

base = 3000 + 0.5 * t
weekly = np.zeros(N_POINTS)
for i in range(N_POINTS):
    dow = dates[i].dayofweek
    if dow <= 4: weekly[i] = 200
    else: weekly[i] = -300

# Double-peak seasonality (summer AC + winter heating)
seasonal = 400 * np.cos(2 * np.pi * (t - 30) / 365) + 200 * np.cos(2 * np.pi * (t - 200) / 365)

holiday = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m, d = dates[i].month, dates[i].day
    if (m == 12 and d >= 24) or (m == 1 and d <= 2): holiday[i] = -500
    elif m == 11 and 22 <= d <= 28: holiday[i] = -300

noise = np.random.normal(0, 100, N_POINTS)
power_mw = base + weekly + seasonal + holiday + noise
power_mw = np.maximum(power_mw, 1000).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'power_mw': power_mw})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,power_mw
0,2022-01-01,2407
1,2022-01-02,2346
2,2022-01-03,3427
3,2022-01-04,3518
4,2022-01-05,3345


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('power_mw Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("power_mw Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

power_mw Statistics:
count     730.000000
mean     3217.556164
std       293.010326
min      2346.000000
25%      3017.000000
50%      3262.000000
75%      3429.750000
max      3822.000000
Name: power_mw, dtype: float64

CV: 0.091
Skewness: -0.476


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:      317.9 | RMSE:      397.5 | MAPE: 10.18%
Seasonal Naive (7)             | MAE:      274.2 | RMSE:      354.0 | MAPE:  9.07%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared   R-Squared         RMSE  Time Taken
Model                                                                            
KernelRidge                      2562.344860 -196.026528  3266.555796    0.056851
MLPRegressor                     2445.236759 -187.018212  3191.006497    0.635504
LinearSVR                        1922.236032 -146.787387  2829.087725    0.014165
GaussianProcessRegressor         1754.780651 -133.906204  2702.985338    0.130117
DummyRegressor                     32.997644   -1.461357   365.102706    0.012959
NuSVR                              29.081240   -1.160095   342.029994    0.047190
QuantileRegressor                  27.768890   -1.059145   333.942146    0.079634
SVR                                24.372718   -0.797901   312.040281    0.043487
DecisionTreeRegressor              13.998688    0.000101   232.705056    0.021069
XGBRegressor                       13.912256    0.006750   231.930098    0.51

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:      126.4 | RMSE:      144.5 | MAPE:  3.62%
FLAML Test (lgbm)              | MAE:      153.3 | RMSE:      188.9 | MAPE:  4.74%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:      279.4 | RMSE:      344.6 | MAPE:  9.43%
SF AutoETS                     | MAE:      306.2 | RMSE:      385.0 | MAPE: 10.36%
SF SeasonalNaive               | MAE:      326.9 | RMSE:      407.2 | MAPE: 11.04%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model        MAE       RMSE      MAPE
      FLAML (lgbm) 126.449088 144.473677  3.619708
 FLAML Test (lgbm) 153.264494 188.864764  4.742421
Seasonal Naive (7) 274.214286 354.036620  9.072328
      SF AutoARIMA 279.363983 344.606535  9.432874
        SF AutoETS 306.176544 384.997687 10.364801
Naive (Last Value) 317.857143 397.524123 10.176273
  SF SeasonalNaive 326.857147 407.215104 11.043918

Top 3:
             model        MAE       RMSE     MAPE
      FLAML (lgbm) 126.449088 144.473677 3.619708
 FLAML Test (lgbm) 153.264494 188.864764 4.742421
Seasonal Naive (7) 274.214286 354.036620 9.072328


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 4.37, Std: 188.81


## Interpretation and Business Insight

- Power demand shows clear weekday/weekend and seasonal (summer/winter) patterns
- Holiday dips are significant and predictable
- Growth trend reflects increasing electrification
- FLAML captures lag patterns; AutoARIMA handles trend + seasonality
- Temperature is the strongest missing predictor

## Limitations

1. Synthetic — real power data depends heavily on weather
2. No temperature/weather features
3. No distinction by sector (residential, commercial, industrial)
4. Daily granularity — real grids need hourly or 15-minute forecasts

## How to Improve This Project

1. Add temperature and weather forecasts as features
2. Use hourly granularity for dispatch-level forecasting
3. Segment by customer class
4. Add renewable generation forecasts
5. Use Chronos-Bolt for foundation-model forecasting

## Production Considerations

- Day-ahead and week-ahead forecasting pipelines
- Integration with SCADA/EMS systems
- Real-time forecast updates as weather changes
- Alert on demand approaching grid capacity

## Common Mistakes

1. Ignoring weather — temperature explains most demand variance
2. Using daily data for intraday dispatch decisions
3. Not handling holiday effects separately
4. Over-fitting on short training windows

## Mini Challenge / Exercises

1. Add a synthetic temperature feature and measure improvement
2. Try weekly aggregation and compare accuracy
3. Build a peak-demand predictor (binary: above/below threshold)
4. Simulate a heat wave and test model response

## Final Summary / Key Takeaways

1. Power demand forecasting is critical for grid operations
2. Weekly and seasonal patterns drive most variation
3. Temperature (not included) is the key missing feature
4. 14-day forecasts support fuel procurement planning
5. Combining ML and statistical approaches gives robust results

In [18]:
import json
metrics = {
    'project': 'Power Demand Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Power Demand Forecasting — Complete ===")

Metrics saved.

=== Power Demand Forecasting — Complete ===
